In [2]:
# Label cleaning: claims_and_billing + denials
from pathlib import Path
import pandas as pd

DATA_DIR = Path("data")
ARTIFACTS_DIR = Path("artifacts")
CLAIMS_PATH = DATA_DIR / "claims_and_billing.csv"
DENIALS_PATH = DATA_DIR / "denials.csv"
OUTPUT_PATH = ARTIFACTS_DIR / "claims_and_billing_cleaned.csv"


def normalize_text(series: pd.Series) -> pd.Series:
    return (
        series.fillna("")
        .astype(str)
        .str.strip()
        .str.lower()
        .str.replace(r"\s+", " ", regex=True)
        .str.replace(r"[.]+$", "", regex=True)
    )


def deduplicate_claims(claims: pd.DataFrame) -> pd.DataFrame:
    dup_mask = claims["claim_id"].duplicated(keep=False)
    if dup_mask.any():
        print(f"[warn] Found {dup_mask.sum()} rows with duplicate claim_id; keeping first per claim_id.")
    return claims.drop_duplicates(subset=["claim_id"], keep="first")


def reconcile_reasons(claims: pd.DataFrame, denials: pd.DataFrame):
    claims = claims.copy()
    denials = denials.copy()

    claims["claim_status"] = claims["claim_status"].str.strip()
    claims = claims[claims["claim_status"].isin(["Paid", "Denied"])]

    claims["denial_reason_norm"] = normalize_text(claims.get("denial_reason", pd.Series(dtype=str)))
    denials["denial_reason_norm"] = normalize_text(denials.get("denial_reason_description", pd.Series(dtype=str)))

    claims_counts = claims.loc[claims["claim_status"] == "Denied", "denial_reason_norm"].value_counts()
    denials_counts = denials["denial_reason_norm"].value_counts()
    mapping = {}
    if len(claims_counts) == len(denials_counts):
        mapping = {long: short for short, long in zip(claims_counts.index, denials_counts.index)}

    denials_small = denials[["claim_id", "denial_reason_norm"]].drop_duplicates("claim_id")
    claims = claims.merge(denials_small, on="claim_id", how="left", suffixes=("", "_denials"))

    def choose_reason(row):
        r_claims = row["denial_reason_norm"]
        r_denials = row.get("denial_reason_norm_denials", "")
        if r_claims:
            return r_claims
        if r_denials:
            return mapping.get(r_denials, r_denials)
        return ""

    claims["denial_reason_clean"] = claims.apply(choose_reason, axis=1)
    claims.loc[claims["claim_status"] != "Denied", "denial_reason_clean"] = ""
    return claims, mapping


def summarize(claims: pd.DataFrame, mapping: dict):
    total = len(claims)
    denied = (claims["claim_status"] == "Denied").sum()
    print(f"[info] total claims: {total}, denied: {denied}, paid: {total - denied}")
    missing_reason = ((claims["claim_status"] == "Denied") & (claims["denial_reason_clean"] == "")).sum()
    print(f"[info] denied with missing clean reason: {missing_reason}")
    if mapping:
        print(f"[info] mapped {len(mapping)} denial descriptions to short forms.")
    top_reasons = claims.loc[claims["claim_status"] == "Denied", "denial_reason_clean"].value_counts().head(10)
    print("\nTop denial reasons (clean):")
    print(top_reasons)


# Load and clean
claims = pd.read_csv(CLAIMS_PATH)
denials = pd.read_csv(DENIALS_PATH)
claims = deduplicate_claims(claims)
claims, mapping = reconcile_reasons(claims, denials)
summarize(claims, mapping)
claims.to_csv(OUTPUT_PATH, index=False)
print(f"\n[done] wrote cleaned claims to {OUTPUT_PATH}")

[warn] Found 10362 rows with duplicate claim_id; keeping first per claim_id.
[info] total claims: 59639, denied: 5998, paid: 53641
[info] denied with missing clean reason: 0
[info] mapped 14 denial descriptions to short forms.

Top denial reasons (clean):
denial_reason_clean
duplicate claim                   472
prior authorization required      457
service not covered               437
timely filing limit exceeded      437
coordination of benefits issue    436
expired or invalid insurance      435
coverage limit exceeded           432
out-of-network provider           425
claim billed to wrong payer       422
invalid place of service          416
Name: count, dtype: int64

[done] wrote cleaned claims to artifacts/claims_and_billing_cleaned.csv


In [5]:
# Build claim-centric training dataset by joining encounters/patients/providers and per-encounter artifacts
from pathlib import Path
import pandas as pd

ARTIFACTS_DIR = Path("artifacts")
DATA_DIR = Path("data")

CLEAN_CLAIMS_PATH = ARTIFACTS_DIR / "claims_and_billing_cleaned.csv"
OUTPUT_JOINED_PATH = ARTIFACTS_DIR / "claims_enriched.csv"

# Load cleaned claims (must exist; run previous cell first)
claims = pd.read_csv(CLEAN_CLAIMS_PATH)
encounters = pd.read_csv(DATA_DIR / "encounters.csv")
diagnoses = pd.read_csv(DATA_DIR / "diagnoses.csv")
procedures = pd.read_csv(DATA_DIR / "procedures.csv")
labs = pd.read_csv(DATA_DIR / "lab_tests.csv")
meds = pd.read_csv(DATA_DIR / "medications.csv")
patients = pd.read_csv(DATA_DIR / "patients.csv")
providers = pd.read_csv(DATA_DIR / "providers.csv")

# Deduplicate keys for safer joins
encounters = encounters.drop_duplicates(subset=["encounter_id"], keep="first")
patients = patients.drop_duplicates(subset=["patient_id"], keep="first")
providers = providers.drop_duplicates(subset=["provider_id"], keep="first")

# Keep only claims that have encounter_id and patient_id (expected 100% coverage)
claims = claims[claims["encounter_id"].notna() & claims["patient_id"].notna()].copy()

# Merge core dimensions
joined = claims.merge(encounters, on="encounter_id", how="left", suffixes=("", "_enc"))
joined = joined.merge(patients, on="patient_id", how="left", suffixes=("", "_pat"))
joined = joined.merge(providers, on="provider_id", how="left", suffixes=("", "_prov"))

# Per-encounter aggregates


def add_counts(df: pd.DataFrame, table: pd.DataFrame, key: str, col: str, prefix: str):
    counts = table.groupby(key).size().rename(f"{prefix}_count")
    distinct = table.groupby(key)[col].nunique().rename(f"{prefix}_distinct")
    return df.merge(counts, left_on=key, right_index=True, how="left").merge(
        distinct, left_on=key, right_index=True, how="left"
    )

joined = add_counts(joined, diagnoses, "encounter_id", "diagnosis_code", "diag")
joined = add_counts(joined, procedures, "encounter_id", "procedure_code", "proc")
joined = add_counts(joined, labs, "encounter_id", "test_code", "lab")
joined = add_counts(joined, meds, "encounter_id", "drug_name", "med")

# Fill count NA with 0
for col in [c for c in joined.columns if c.endswith("_count") or c.endswith("_distinct")]:
    joined[col] = joined[col].fillna(0).astype(int)

# Labels
joined["label_denied"] = (joined["claim_status"] == "Denied").astype(int)
joined["label_reason"] = joined["denial_reason_clean"].fillna("")

# Drop leakage/PII columns not suitable for modeling
leak_cols = {
    "label_reason",
    "denial_reason",
    "denial_reason_norm",
    "denial_reason_norm_denials",
    "denial_reason_clean",
    "claim_status",
    "paid_amount",  # post-outcome
}
# Keep patient_id for grouped splitting downstream; drop other identifiers/PII
id_cols = {
    "billing_id",
    "claim_id",
    "encounter_id",
    "provider_id",
    "patient_id_enc",
    "first_name",
    "last_name",
    "address",
    "city",
    "state",
    "zip",
    "phone",
    "email",
    "registration_date",
    "name",
    "department_prov",
    "contact_info",
    "email_prov",
}
joined = joined.drop(columns=list(leak_cols | id_cols), errors="ignore")

# Basic sanity checks
print("joined rows", len(joined))
print("label_denied distribution:", joined["label_denied"].value_counts().to_dict())

# Save
OUTPUT_JOINED_PATH.parent.mkdir(parents=True, exist_ok=True)
joined.to_csv(OUTPUT_JOINED_PATH, index=False)
print(f"[done] wrote enriched claims to {OUTPUT_JOINED_PATH}")



joined rows 59639
label_denied distribution: {0: 53641, 1: 5998}
[done] wrote enriched claims to artifacts/claims_enriched.csv


In [6]:
# Split into train/eval sets (grouped by patient to avoid leakage)
from pathlib import Path
import pandas as pd
from sklearn.model_selection import GroupShuffleSplit

ARTIFACTS_DIR = Path("artifacts")
INPUT_PATH = ARTIFACTS_DIR / "claims_enriched.csv"
TRAIN_PATH = ARTIFACTS_DIR / "claims_enriched_train.csv"
EVAL_PATH = ARTIFACTS_DIR / "eval" / "claims_enriched_eval.csv"

EVAL_PATH.parent.mkdir(parents=True, exist_ok=True)

# Load enriched data
claims_enriched = pd.read_csv(INPUT_PATH)

# Grouped split by patient_id to prevent leakage
groups = claims_enriched["patient_id"]
splitter = GroupShuffleSplit(test_size=0.2, n_splits=1, random_state=42)
train_idx, eval_idx = next(splitter.split(claims_enriched, groups=groups))

train_df = claims_enriched.iloc[train_idx].reset_index(drop=True)
eval_df = claims_enriched.iloc[eval_idx].reset_index(drop=True)

print("Train rows:", len(train_df), "Eval rows:", len(eval_df))
print("Train label distribution:", train_df["label_denied"].value_counts().to_dict())
print("Eval label distribution:", eval_df["label_denied"].value_counts().to_dict())

train_df.to_csv(TRAIN_PATH, index=False)
eval_df.to_csv(EVAL_PATH, index=False)
print(f"[done] wrote train -> {TRAIN_PATH}\n[done] wrote eval -> {EVAL_PATH}")



Train rows: 47744 Eval rows: 11895
Train label distribution: {0: 42933, 1: 4811}
Eval label distribution: {0: 10708, 1: 1187}
[done] wrote train -> artifacts/claims_enriched_train.csv
[done] wrote eval -> artifacts/eval/claims_enriched_eval.csv
